In [1]:
%%writefile configs.yaml
model:
    hidden_dim: 768
    num_dit_blocks: 12
    head_dim: 64
    num_heads: 12
    text_dim:
    ffn_multiplier: 4
    dropout: 0.0
    prompt_conv_channels: [128, 256, 512]
    prompt_conv_kernel: 5
    mel_dim: 100
    vocab_size: 256
    text_layers: 10
    pe_attn_head:
    text_mask_padding:

data:
    dataset_dirs:
        - "/kaggle/input/datasets/huutrank4ds/viflow-cleaned-part-1"
        - "/kaggle/input/datasets/huutrank4ds/viflow-cleaned-part-2"
        - "/kaggle/input/datasets/huutrank4ds/viflow-cleaned-part-5"
    val_sample_path: "/kaggle/input/datasets/huutrank4ds/validation-sample-viflow/mixed_speaker_samples.csv"

mel:
    sample_rate: 24000
    n_fft: 1024
    hop_length: 256
    win_length: 1024
    n_mels: 100
    fmin: 0
    fmax: 12000.0
    power: 1.0
    mel_scale: "slaney"
    norm: "slaney"
    log_clip_min: 1.0e-5

train:
    port: '12355'
    batch_size: 32
    num_workers: 4
    learning_rate: 2.0e-4
    weight_decay: 1.0e-2
    grad_clip_norm: 1.0
    num_epochs: 100
    save_interval: 200
    grad_clip: 1
    checkpoint_dir: "checkpoints"
    log_dir: "logs"
    resume_path: "viflow.pt"
    use_amp: True

inference:
    ode_steps: 16


Writing configs.yaml


In [2]:
%%writefile dataset.py
# --- 1. Standard Libraries ---
import io
import os
import glob
import yaml
import math
from pathlib import Path
from typing import List, Optional

# --- 2. Data & Math Libraries ---
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from librosa.filters import mel as librosa_mel_fn

# --- 3. PyTorch Ecosystem ---
import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from einops import rearrange

class VietnameseTokenizer:
    """
    Tokenizer tĩnh dựa trên bảng chữ cái tiếng Việt.
    -
    Static tokenizer based on the Vietnamese alphabet.
    """
    def __init__(self):
        self.pad_token = "<pad>"
        self.unk_token = "<unk>"
        
        # Bảng chữ cái tiếng Việt đầy đủ (bao gồm cả dấu thanh) + các dấu câu cơ bản
        raw_chars = " !\"#$%&'(),-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz«»ÀÁÂÉÊÍÔÚÜÝàáâãäçèéêëìíîòóôõöøùúüýĂăćĐđęĩıłŌōœšũƠơƯưșạẢảẤấầẨẩẫẬậẮắẰằẳẵặẹẻẽẾếỀềểễệỉỊịỌọỏỐốỒồỔổỗộớỜờỞởỡỢợụỦủỨứỪừửữỰựỳỵỷỹ–…"
        # Chuẩn hóa: Chuyển về chữ thường -> Lọc các ký tự duy nhất -> Sắp xếp để đảm bảo ID nhất quán
        cleaned_chars = "".join(sorted(list(set(raw_chars.lower()))))

        self.vocab = [self.pad_token, self.unk_token] + list(cleaned_chars)
        self.symbol_to_id = {s: i for i, s in enumerate(self.vocab)}

    @property
    def pad_id(self):
        return 0

    @property
    def vocab_size(self):
        return len(self.vocab)

    def encode_batch(self, texts):
        # 1. Chuyển đổi text thành list các Tensor (bắt buộc cho pad_sequence)
        encoded_tensors = [
            torch.tensor([self.symbol_to_id.get(char, 1) for char in text.lower()], dtype=torch.long) 
            for text in texts
        ]
        
        # 2. Tính lengths trước khi padding
        lengths = torch.tensor([len(x) for x in encoded_tensors], dtype=torch.long)
        
        # 3. Sử dụng pad_sequence
        # batch_first=True để output có shape [Batch, Max_Len] giống code cũ của bạn
        ids = pad_sequence(
            encoded_tensors, 
            batch_first=True, 
            padding_value=self.pad_id # Giả định bạn đã định nghĩa self.pad_id = 0
        )
        
        return ids, lengths

class ViFlowDataset(Dataset):
    def __init__(
        self, 
        dataset_dirs: List[str], 
        sample_rate: int = 24000,
        include_csv_path: Optional[str] = None,
        exclude_csv_path: Optional[str] = None
    ):
        self.sample_rate = sample_rate
        
        # 1. Thu thập danh sách file parquet
        parquet_files = []
        for d in dataset_dirs:
            parquet_files.extend(glob.glob(os.path.join(d, "**/*.parquet"), recursive=True))

        # 2. Xây dựng Metadata thô từ Parquet
        meta_list = []
        for f in parquet_files:
            # Chỉ đọc các cột cần thiết để tiết kiệm RAM
            temp_df = pd.read_parquet(f, columns=['text', 'speaker', '_id'])
            temp_df['file_path'] = f
            meta_list.append(temp_df)
        
        full_metadata = pd.concat(meta_list, ignore_index=True)

        # 3. LOGIC BỘ LỌC (CSV Include/Exclude)
        self.metadata = self._filter_metadata(
            full_metadata, include_csv_path, exclude_csv_path
        )
        
        print(f"✅ Dataset ready: {len(self.metadata)} samples (Filters applied via CSV)")
        
    def _filter_metadata(self, df, include_path, exclude_path):
        """Lọc dữ liệu dựa trên file CSV chứa cột 'speaker' và '_id'"""
        
        # Tạo khóa định danh duy nhất cho từng mẫu
        df['filter_key'] = df['speaker'].astype(str) + "__" + df['_id'].astype(str)

        def get_keys_from_csv(path):
            if not path or not os.path.exists(path):
                return None
            # Đọc CSV và tạo set key "speaker__id"
            filter_df = pd.read_csv(path)
            if 'speaker' not in filter_df.columns or '_id' not in filter_df.columns:
                raise ValueError(f"File {path} thiếu cột 'speaker' hoặc '_id'!")
            
            return set(filter_df['speaker'].astype(str) + "__" + filter_df['_id'].astype(str))

        include_set = get_keys_from_csv(include_path)
        exclude_set = get_keys_from_csv(exclude_path)

        # A. Whitelist: Chỉ giữ lại các mẫu trong file include
        if include_set is not None:
            df = df[df['filter_key'].isin(include_set)]
        
        # B. Blacklist: Loại bỏ các mẫu trong file exclude
        if exclude_set is not None:
            df = df[~df['filter_key'].isin(exclude_set)]

        return df.drop(columns=['filter_key']).reset_index(drop=True)

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row = self.metadata.iloc[idx]
        file_path = row['file_path']
        audio_id = str(row['_id'])
        text = str(row['text'])
        speaker_id = str(row['speaker'])

        try:
            # Truy xuất "lười" bằng pyarrow (nhanh và ít RAM)
            table = pq.read_table(
                file_path, columns=['audio'], 
                filters=[('_id', '==', audio_id)]
            )
            
            if len(table) == 0:
                return self.__getitem__((idx + 1) % len(self))
                
            audio_bytes = table.column('audio')[0].as_py()
            wav, sr = torchaudio.load(io.BytesIO(audio_bytes))
            
            # Tiền xử lý audio
            if sr != self.sample_rate:
                wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
            if wav.shape[0] > 1:
                wav = wav.mean(0, keepdim=True)
            wav = wav.squeeze(0)

            return {
                "text": text,
                "wav": wav,
                "speaker": speaker_id
            }
        except Exception:
            return self.__getitem__((idx + 1) % len(self))


class SpeechProcessor:
    def __init__(self, tokenizer, mel_cfg, device="cuda"):
        self.tokenizer = tokenizer
        self.device = device
        
        # 1. Lấy thông số từ config
        self.target_sr = mel_cfg['sample_rate']
        self.hop_size = mel_cfg['hop_length']
        self.win_size = mel_cfg['win_length']
        self.n_fft = mel_cfg['n_fft']
        self.n_mels = mel_cfg['n_mels']
        self.mel_pad = math.log(mel_cfg['log_clip_min'])

        # 2. Khởi tạo Ma trận Mel & Window TRỰC TIẾP trên device
        mel_basis = librosa_mel_fn(
            sr=self.target_sr, 
            n_fft=self.n_fft, 
            n_mels=self.n_mels, 
            fmin=mel_cfg.get('fmin', 0), 
            fmax=mel_cfg.get('fmax', 12000)
        )
        self.mel_basis = torch.from_numpy(mel_basis).float().to(device)
        self.window = torch.hann_window(self.win_size).to(device)

    def _compute_mel(self, wavs_padded):
        """Tính Mel chuẩn BigVGAN cho cả Batch"""
        # Manual Padding theo chuẩn BigVGAN để căn chỉnh pha
        pad_size = (self.n_fft - self.hop_size) // 2
        x = F.pad(wavs_padded.unsqueeze(1), (pad_size, pad_size), mode='reflect').squeeze(1)

        # STFT Batch Processing
        spec = torch.stft(
            x, self.n_fft, hop_length=self.hop_size, win_length=self.win_size, 
            window=self.window, center=False, pad_mode='reflect', 
            normalized=False, onesided=True, return_complex=True
        )
        
        spec = torch.abs(spec) + 1e-9
        mel = torch.matmul(self.mel_basis, spec)
        # Sử dụng log tự nhiên và clamp chuẩn F5-TTS
        log_mel = torch.log(torch.clamp(mel, min=1e-5))
        
        return log_mel.transpose(1, 2) # [B, T_max, n_mels]

    def process_batch(self, batch):
        """
        Xử lý Batch: Trả về List các tensor để chuẩn bị cho việc nối chuỗi Unified
        """
        # --- A. Chuẩn bị dữ liệu ---
        texts = [item['text'] for item in batch]
        
        # Đảm bảo wav là mono và đúng sample rate trước khi đưa vào batch
        wavs_list = []
        for item in batch:
            w = item['wav'].to(self.device)
            wavs_list.append(w)

        # 1. Tính Mel hàng loạt trên GPU
        wavs_padded = pad_sequence(wavs_list, batch_first=True)
        # Tính độ dài frame thực tế của mỗi mẫu
        wav_lens = torch.tensor([w.size(-1) for w in wavs_list], device=self.device)
        mel_lens = torch.floor(wav_lens.float() / self.hop_size).long() # (b)
        
        mels_padded = self._compute_mel(wavs_padded) # (b, t, d)
        mel_mask = (torch.arange(mels_padded.size(1), device=self.device).unsqueeze(0) >= mel_lens.unsqueeze(1)) # (b, t)
        mels_padded = mels_padded.masked_fill(mel_mask.unsqueeze(-1), self.mel_pad)

        # --- B. Xử lý Text ---
        text_ids, text_lens = self.tokenizer.encode_batch(texts)
        text_ids = text_ids.to(self.device)
        text_lens = text_lens.to(self.device)

        # --- D. Kết quả trả về ---
        return {
            "text_ids": text_ids,
            "text_lens": text_lens,
            "target_mels": mels_padded,
            "target_mel_lens": mel_lens,
            "mel_mask": mel_mask
        }


def identity_collate(batch):
    return batch

Writing dataset.py


In [3]:
%%writefile text_embedding.py
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

def precompute_freqs_cis(dim: int, end: int, theta: float = 10000.0, theta_rescale_factor: float = 1.0):
    if theta_rescale_factor != 1.0:
        theta *= theta_rescale_factor ** (dim / (dim - 2))
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(end, device=freqs.device)
    angles = torch.outer(t, freqs).float()
    return torch.cat([angles.cos(), angles.sin()], dim=-1)


def get_pos_embed_indices(start, length, max_pos, scale=1.0):
    scale = scale * torch.ones_like(start, dtype=torch.float32)
    pos = (
        start.unsqueeze(1)
        + (torch.arange(length, device=start.device, dtype=torch.float32).unsqueeze(0) * scale.unsqueeze(1)).long()
    )
    pos = torch.where(pos < max_pos, pos, max_pos - 1)
    return pos


class GRN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.gamma = nn.Parameter(torch.zeros(1, 1, dim))
        self.beta = nn.Parameter(torch.zeros(1, 1, dim))

    def forward(self, x):
        Gx = torch.norm(x, p=2, dim=1, keepdim=True)
        Nx = Gx / (Gx.mean(dim=-1, keepdim=True) + 1e-6)
        return self.gamma * (x * Nx) + self.beta + x

    
class ConvNeXtV2Block(nn.Module):
    def __init__(self, dim, conv_mult=2, kernel_size=7, dilation=1):
        super().__init__()
        # 1. Tự động tính toán intermediate_dim dựa trên hệ số nhân
        self.intermediate_dim = dim * conv_mult
        # 2. Depthwise Conv
        padding = (kernel_size - 1) * dilation // 2
        self.dwconv = nn.Conv1d(
            dim, dim, 
            kernel_size=kernel_size, 
            padding=padding, 
            groups=dim, 
            dilation=dilation
        )
        # 3. LayerNorm & Pointwise mappings
        self.norm = nn.LayerNorm(dim, eps=1e-6)
        # Linear 1: Phình to ra (D -> D * conv_mult)
        self.pwconv1 = nn.Linear(dim, self.intermediate_dim)
        self.act = nn.GELU()
        # 4. GRN (Global Response Normalization) 
        self.grn = GRN(self.intermediate_dim)
        # Linear 2: Thu nhỏ lại (D * conv_mult -> D)
        self.pwconv2 = nn.Linear(self.intermediate_dim, dim)

    def forward(self, x):
        residual = x
        
        # Step 1: Depthwise Conv (Bắt ngữ cảnh thời gian)
        x = x.transpose(1, 2)
        x = self.dwconv(x)
        x = x.transpose(1, 2)
        
        # Step 2: MLP-style (Bắt đặc trưng kênh)
        x = self.norm(x)
        x = self.pwconv1(x)
        x = self.act(x)
        x = self.grn(x)
        x = self.pwconv2(x)
        
        return residual + x


class TextEmbedding(nn.Module):
    freqs_cis: torch.Tensor
    def __init__(self, vocab_size, text_dim, padding_idx=0, n_layers=4, n_mult=2, kernel_size=7, mask_padding=True):
        super().__init__()
        
        # 1. Token Embedding (Sử dụng 0 làm filler/padding token)
        self.embedding = nn.Embedding(vocab_size, text_dim, padding_idx=padding_idx)   
        self.mask_padding = mask_padding
        
        if n_layers > 0:
            self.extra_modeling = True
            self.max_pos = 4096
            self.register_buffer("freqs_cis", precompute_freqs_cis(text_dim, self.max_pos), persistent=False)
            self.text_blocks = nn.ModuleList([
                ConvNeXtV2Block(dim=text_dim, conv_mult=n_mult, kernel_size=kernel_size) 
                for _ in range(n_layers)
            ])
        else:
            self.extra_modeling = False

    def forward(self, text_ids, seq_len):
        batch, text_len = text_ids.shape
        text_ids = F.pad(text_ids, (0, seq_len - text_len), value=0)

        if self.mask_padding:
            text_mask = (text_ids == 0).unsqueeze(-1) # (b, t, 1)

        x = self.embedding(text_ids) # (b, seq_len) -> (b, seq_len, dim)
        
        if self.extra_modeling:  
            batch_start = torch.zeros((batch,), dtype=torch.long, device=text_ids.device)
            pos_idx = get_pos_embed_indices(batch_start, seq_len, max_pos=self.max_pos)
            
            x = x + self.freqs_cis[pos_idx]

            if self.mask_padding:
                x = x.masked_fill(text_mask, 0.0)
                for block in self.text_blocks:
                    x = block(x)
                    x = x.masked_fill(text_mask, 0.0)
            else:
                for block in self.text_blocks:
                    x = block(x)
        return x

Writing text_embedding.py


In [4]:
%%writefile timestep_embedding.py
import math
import torch
import torch.nn as nn

class SinusoidalPosEmb(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, x, scale=1000):
        device = x.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device).float() * -emb)
        emb = scale * x.unsqueeze(1) * emb.unsqueeze(0)
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

class TimestepEmbedding(nn.Module):
    def __init__(self, hidden_dim: int, fourier_dim: int = 256):
        """
        Lớp bọc (Wrapper) để đưa timestep qua Sinusoidal và MLP.
        """
        super().__init__()
        # 1. Lớp mã hóa lượng giác "chính chủ" F5-TTS
        self.sinusoidal_emb = SinusoidalPosEmb(fourier_dim)
        
        # 2. Mạng MLP để map fourier_dim (256) lên hidden_dim (768)
        self.mlp = nn.Sequential(
            nn.Linear(fourier_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        """
        t: Tensor [B] hoặc [B, 1] nằm trong khoảng [0, 1]
        """
        # Nếu t là [B, 1], bóp về [B] để khớp với logic của SinusoidalPosEmb
        if t.dim() == 2:
            t = t.squeeze(-1)
            
        # Bước 1: Tạo chữ ký sin/cos (256 dims)
        # scale=1000 đã được định nghĩa mặc định trong SinusoidalPosEmb.forward
        temb = self.sinusoidal_emb(t)
        
        # Bước 2: Phóng đại qua MLP để ra vector điều kiện cuối cùng
        return self.mlp(temb)

Writing timestep_embedding.py


In [5]:
%%writefile dit_layers.py
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional
from einops import rearrange 

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm_x = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm_x * self.scale

class AdaLayerNorm(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.silu = nn.SiLU()
        self.linear = nn.Linear(dim, dim * 6)

        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)

    def forward(self, x, emb=None):
        ada_params = self.linear(self.silu(emb)) # (b, d) -> (b, 6d)
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = torch.chunk(ada_params, 6, dim=1)

        x = self.norm(x) * (1 + scale_msa.unsqueeze(1)) + shift_msa.unsqueeze(1)
        return x, gate_msa, shift_mlp, scale_mlp, gate_mlp

class AdaLayerNorm_Final(nn.Module):
    def __init__(self, dim):
        super().__init__()

        self.silu = nn.SiLU()
        self.linear = nn.Linear(dim, dim * 2)

        self.norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)

    def forward(self, x, emb):
        ada_params = self.linear(self.silu(emb))
        scale, shift = torch.chunk(ada_params, 2, dim=1)

        x = self.norm(x) * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)
        return x

class ConvPositionEmbedding(nn.Module):
    def __init__(self, dim, kernel_size=31):
        super().__init__()
        # Thường dùng Depthwise Conv để tiết kiệm tham số và tập trung vào từng kênh
        self.conv = nn.Conv1d(
            dim, dim, kernel_size, 
            padding=kernel_size // 2, 
            groups=dim
        )

    def forward(self, x):
        # x shape: [B, L, D]
        x = x.transpose(1, 2) # Chuyển sang [B, D, L] để chạy Conv
        x = self.conv(x)
        return x.transpose(1, 2)
        
class RotaryEmbedding(nn.Module):
    inv_freq: torch.Tensor
    def __init__(self, dim, theta=10000):
        super().__init__()
        self.dim = dim
        self.theta = theta
        inv_freq = 1.0 / (self.theta ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, seq_len, device):
        # Tự sinh cache cos, sin dựa trên độ dài chuỗi
        t = torch.arange(seq_len, device=device).type_as(self.inv_freq)
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        # [T, 1, D] để broadcast mượt mà với shape [B, T, H, D]
        return emb.cos()[:, None, :], emb.sin()[:, None, :]

    def rotate_queries_and_keys(self, q, k, seq_dim=1):
        # 1. Lấy thông tin thiết bị và độ dài chuỗi từ q
        device = q.device
        seq_len = q.shape[seq_dim]
        
        # 2. Tính cos/sin một lần duy nhất cho cả lượt
        cos, sin = self.forward(seq_len, device)
        
        # 3. Xoay cả hai bằng công thức: x * cos + rotate_half(x) * sin
        q = (q * cos) + (rotate_half(q) * sin)
        k = (k * cos) + (rotate_half(k) * sin)
        
        return q, k

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

class MultiHeadSelfAttention(nn.Module):
    def __init__(
        self, 
        hidden_dim: int, 
        head_dim:int,
        num_heads: int, 
        qk_norm: bool = False, 
        pe_attn_head: Optional[int] = None, # Số lượng head được xoay RoPE
        dropout: float = 0.0
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = head_dim
        self.inner_dim = head_dim * num_heads
        self.qk_norm = qk_norm
        
        # pn: Nếu không truyền, mặc định xoay tất cả heads
        self.pe_attn_head = pe_attn_head if pe_attn_head is not None else num_heads
        
        self.qkv = nn.Linear(hidden_dim, self.inner_dim * 3, bias=False)
        
        if qk_norm:
            self.q_norm = RMSNorm(self.head_dim, eps=1e-6)
            self.k_norm = RMSNorm(self.head_dim, eps=1e-6)
            
        self.to_out = nn.Sequential(
            nn.Linear(self.inner_dim, hidden_dim, bias=False),
            nn.Dropout(dropout)
        )

    def forward(self, x, mask=None, rope=None):
        # x shape: [B, T, C]
        batch_size, seq_len, _ = x.shape
        
        # --- BƯỚC 1: QKV Projection & Rearrange ---
        qkv = self.qkv(x).chunk(3, dim=-1)
        # Shape: [B, T, H, D]
        q, k, v = map(lambda t: rearrange(t, "b t (h d) -> b t h d", h=self.num_heads), qkv)

        # --- BƯỚC 2: QK Norm (Tùy chọn) ---
        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        # --- BƯỚC 3: RoPE (Cơ chế Partial Heads - pe_attn_head) ---
        if rope is not None:
            pn = self.pe_attn_head
            if pn < self.num_heads:
                # Tách heads: [B, T, 0:pn, D] và [B, T, pn:H, D]
                q_rope, q_pass = q[:, :, :pn, :], q[:, :, pn:, :]
                k_rope, k_pass = k[:, :, :pn, :], k[:, :, pn:, :]
                
                # Chỉ xoay nhóm pn heads đầu tiên
                q_rope, k_rope = rope.rotate_queries_and_keys(q_rope, k_rope)
                
                # Nối lại theo chiều Head (dim=2 trong shape b t h d)
                q = torch.cat((q_rope, q_pass), dim=2)
                k = torch.cat((k_rope, k_pass), dim=2)
            else:
                # Xoay toàn bộ heads
                q, k = rope.rotate_queries_and_keys(q, k)

        # --- BƯỚC 4: Masking & Attention (SDPA) ---
        # Chuyển sang [B, H, T, D] để SDPA hiểu đúng
        q, k, v = map(lambda t: t.transpose(1, 2), (q, k, v))
        
        attn_mask = mask.unsqueeze(1).unsqueeze(1) if mask is not None else None
        
        out = F.scaled_dot_product_attention(
            q, k, v, 
            attn_mask=attn_mask,
            dropout_p=0.0, # Luôn bằng 0 cho Attention Core
            is_causal=False
        )

        # --- BƯỚC 5: Kết thúc & Clean Padding ---
        out = rearrange(out, "b h t d -> b t (h d)")
        out = self.to_out(out) # Linear + Dropout

        if mask is not None:
            # Ép các vị trí Padding về 0 để tránh rò rỉ thông tin
            out = out.masked_fill(~mask.unsqueeze(-1), 0.0)

        return out

class DiTBlock(nn.Module):
    def __init__(self, dim, heads, head_dim, ff_mult=4, dropout=0.1, qk_norm=None, pe_attn_head=None):
        super().__init__()
        self.attn_norm = AdaLayerNorm(dim)
        self.attn = MultiHeadSelfAttention(
            hidden_dim=dim,
            head_dim=head_dim,
            num_heads=heads,
            qk_norm=False,
            pe_attn_head=pe_attn_head,
            dropout=dropout
        )
        
        self.ff_norm = nn.LayerNorm(dim, elementwise_affine=False, eps=1e-6)
        self.ff = FeedForward(hidden_dim=dim, multiplier=ff_mult, dropout=dropout, approximate="tanh")

    def forward(self, x, t, mask=None, rope=None):
        norm, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.attn_norm(x, emb=t)

        # attention
        attn_output = self.attn(x=norm, mask=mask, rope=rope)

        # process attention output for input x
        x = x + gate_msa.unsqueeze(1) * attn_output

        norm = self.ff_norm(x) * (1 + scale_mlp.unsqueeze(1)) + shift_mlp.unsqueeze(1)

        ff_output = self.ff(norm)
        x = x + gate_mlp.unsqueeze(1) * ff_output
        return x
    

class FeedForward(nn.Module):
    """Transformer feed-forward network with GeLU."""

    def __init__(self, hidden_dim: int, multiplier: int = 4, dropout: float = 0.0, approximate='none'):
        super().__init__()
        inner = hidden_dim * multiplier
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, inner),
            nn.GELU(approximate=approximate),
            nn.Dropout(dropout),
            nn.Linear(inner, hidden_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

Writing dit_layers.py


In [6]:
%%writefile engine.py
import torch
import torch.nn.functional as F
import numpy as np

class ViFlowEngine:
    def __init__(self, sigma_min: float = 1e-5):
        # F5-TTS dùng 1e-5 để ổn định hơn tại t=0 (điểm bắt đầu của nhiễu)
        self.sigma_min = sigma_min

    # --- PHẦN 1: TRAINING LOGIC ---
    def get_train_batch(self, x1, mel_lens, mel_mask, min_p=0.2, max_p=0.4):
        """
        x1: [B, N, 100] - Mel sạch nguyên bản từ Dataloader
        mel_lens: [B] - Độ dài thực tế của từng mẫu trong batch
        """
        batch_size, max_seq_len, dim = x1.size()
        device = x1.device

        p_ratio = torch.rand(batch_size, device=device) * (max_p - min_p) + min_p
        p_len = (p_ratio * mel_lens).long()
        
        indices = torch.arange(max_seq_len, device=device).unsqueeze(0) # [1, T]
        target_mask = (~mel_mask) & (indices >= p_len.unsqueeze(1))
        
        # Ép về float và mở rộng chiều để nhân ma trận
        m_exp = target_mask.unsqueeze(-1).float()

        # Khởi tạo Flow Matching
        t = torch.rand(batch_size, device=device)
        t_exp = t.view(-1, 1, 1)
        x0 = torch.randn_like(x1) # Nhiễu Gaussian trắng

        # Tạo x_t (State) theo công thức Infilling
        # Vùng Prompt: Giữ nguyên x1
        # Vùng Target: (1 - (1-sigma)*t)*x0 + t*x1
        xt_target = (1.0 - (1.0 - self.sigma_min) * t_exp) * x0 + t_exp * x1
        xt = (1.0 - m_exp) * x1 + m_exp * xt_target

        # Tạo cond (Reference Mel)
        # Vùng Prompt: Giữ x1. Vùng Target: Để số 0
        cond = (1.0 - m_exp) * x1 

        # Vận tốc lý tưởng ut (Ground Truth)
        # ut = x1 - (1 - sigma_min) * x0
        ut = x1 - (1.0 - self.sigma_min) * x0
        return xt, cond, ut, t, target_mask

    
    def compute_loss(self, v_pred, v_target, target_mask):
        """
        v_pred:   [B, N, 100] - Đầu ra từ DiT (Velocity field dự đoán)
        v_target: [B, N, 100] - ut từ Engine (Vận tốc lý tưởng x1 - (1-sigma)*x0)
        target_mask: [B, N]   - Mask từ Engine (1: Vùng Target cần học, 0: Prompt & Padding)
        """
        # 1. Đưa mask về shape [B, N, 1] để broadcast với Mel channels
        if target_mask.ndim == 2:
            target_mask = target_mask.unsqueeze(-1)
    
        # 2. Tính Bình phương sai số (MSE) cho từng điểm dữ liệu
        # Không dùng reduction='mean' ngay để ta chủ động lọc mask
        mse = F.mse_loss(v_pred, v_target, reduction="none") # [B, N, 100]
    
        # 3. Lọc qua mask: Triệt tiêu hoàn toàn Loss tại vùng Prompt và Padding
        masked_mse = mse * target_mask
    
        # 4. Tính Loss trung bình trên các frames HỢP LỆ
        # Tổng loss chia cho (tổng số frames được học * số channels)
        # Thêm 1e-8 để tránh chia cho 0 nếu batch toàn padding (hiếm gặp)
        num_active_elements = target_mask.sum() * v_target.size(-1)
        loss = masked_mse.sum() / (num_active_elements + 1e-8)
        return loss

    
    # --- PHẦN 2: INFERENCE LOGIC ---
    @torch.no_grad()
    def solve_ode(self, model, x0, steps, cond, text_ids, mask, sway_coef=-1.0, cfg_scale=1.5):
        xt = x0
        device = x0.device
        
        # Lịch trình Sway
        rho = torch.linspace(0, 1, steps + 1, device=device)
        t_schedule = rho + (sway_coef * torch.sin(2 * np.pi * rho) / (2 * np.pi))

        for i in range(steps):
            t_curr = t_schedule[i]
            t_next = t_schedule[i+1]
            dt = t_next - t_curr
            t_tensor = torch.full((x0.size(0),), t_curr.item(), device=device)
            
            if cfg_scale > 1.0:
                # 1. Luồng có điều kiện đầy đủ
                v_cond = model(x=xt, cond=cond, text_ids=text_ids, t=t_tensor, mask=mask, drop_audio_cond=False)
                
                # 2. Luồng không điều kiện (Null condition)
                # Ta drop audio bằng flag và drop text bằng cách đưa về 0
                v_uncond = model(
                    x=xt, 
                    cond=cond,
                    text_ids=text_ids, 
                    t=t_tensor, 
                    mask=mask, 
                    drop_audio_cond=True
                )
                
                vt = v_uncond + cfg_scale * (v_cond - v_uncond)
            else:
                vt = model(x=xt, cond=cond, text_ids=text_ids, t=t_tensor, mask=mask)
            
            xt = xt + vt * dt
        return xt

Writing engine.py


In [7]:
%%writefile models.py
from typing import List, Sequence

import torch
import torch.nn as nn
import torch.nn.functional as F
from dit_layers import (
    DiTBlock,
    ConvPositionEmbedding,
    RotaryEmbedding,
    AdaLayerNorm_Final
)

from text_embedding import TextEmbedding
from timestep_embedding import TimestepEmbedding

class InputEmbedding(nn.Module):
    def __init__(self, mel_dim, text_dim, out_dim):
        super().__init__()
        self.proj = nn.Linear(mel_dim * 2 + text_dim, out_dim)
        self.conv_pos_embed = ConvPositionEmbedding(dim=out_dim, kernel_size=31)

    def forward(self, x, cond, text_embed, drop_audio_cond=False):
        if drop_audio_cond:
            cond = torch.zeros_like(cond)
        
        combined = torch.cat((x, cond, text_embed), dim=-1) # [B, N, D*3]
        
        x = self.proj(combined) # [B, N, out_dim]
        return self.conv_pos_embed(x) + x


class ViFlowOTCFM(nn.Module):
    def __init__(
        self,
        dim=768,
        depth=22,
        head_dim=64,
        heads=12,
        text_dim=None,
        text_mask_padding=True,
        mel_dim=100,
        vocab_size=256,
        text_layers=10,
        pe_attn_head=None,
        dropout=0.1
    ):
        super().__init__()
        self.dim = dim
        if text_dim is None:
            text_dim = mel_dim
        
        # 1. Mã hóa văn bản (Text Encoder nội bộ)
        # Text dim chốt cứng 100 để khớp với InputEmbedding của sếp
        self.text_encoder = TextEmbedding(
            vocab_size=vocab_size, 
            text_dim=text_dim,
            n_layers=text_layers,
            mask_padding=text_mask_padding
        )
        
        # 2. Điều kiện thời gian (Flow Matching Timestep)
        self.time_embed = TimestepEmbedding(hidden_dim=dim)
        
        # 3. Lớp hòa trộn đầu vào (Combined: Noisy Mel + Ref Mel + Text)
        self.input_embed = InputEmbedding(
            mel_dim=mel_dim, 
            text_dim=text_dim, 
            out_dim=dim
        )
        
        # 4. Rotary Positional Embedding (RoPE) - Khởi tạo 1 lần dùng chung
        self.rope = RotaryEmbedding(dim // heads)
        
        # 5. Danh sách các DiT Blocks (Trái tim của mô hình)
        self.transformer_blocks = nn.ModuleList([
            DiTBlock(
                dim=dim, 
                head_dim=head_dim,
                heads=heads, 
                dropout=dropout,
                pe_attn_head=pe_attn_head
            ) for _ in range(depth)
        ])
        
        # 6. Lớp chuẩn hóa cuối cùng (Adaptive) và Projection
        self.final_norm = AdaLayerNorm_Final(dim)
        self.final_proj = nn.Linear(dim, mel_dim)
        self.initialize_weights()

    def initialize_weights(self):
        for block in self.transformer_blocks:
            # attn_norm.linear là lớp sinh ra 6 tham số điều kiện
            nn.init.constant_(block.attn_norm.linear.weight, 0)
            nn.init.constant_(block.attn_norm.linear.bias, 0)

        # 2. Khởi tạo cho AdaLayerNorm_Final (norm_out)
        nn.init.constant_(self.final_norm.linear.weight, 0)
        nn.init.constant_(self.final_norm.linear.bias, 0)

        # 3. Khởi tạo cho Lớp Projection cuối cùng (proj_out)
        nn.init.constant_(self.final_proj.weight, 0)
        nn.init.constant_(self.final_proj.bias, 0)

    def forward(self, x, cond, text_ids, t, mask=None, drop_audio_cond=False):
        """
        x: Noisy Mel [B, T, 100]
        cond: Reference Mel [B, T, 100]
        text_ids: Token IDs [B, T]
        t: Timestep [B] (từ 0 đến 1)
        mask: Padding Mask [B, T]
        """
        max_mel_len = x.shape[1]
        # Mã hóa văn bản
        text_embed = self.text_encoder(text_ids, max_mel_len) # [b, t, d]
        # Tạo vector thời gian
        t_emb = self.time_embed(t) # [B, dim]

        # Kết hợp (Noisy, Ref, Text) -> ConvPosEmbed -> Proj
        x = self.input_embed(x, cond, text_embed, drop_audio_cond=drop_audio_cond) # [B, T, dim]
        
        # --- TRANSFORMER BLOCKS ---
        for block in self.transformer_blocks:
            # Truyền x, t_emb, mask và rope vào từng block
            x = block(x, t_emb, mask=mask, rope=self.rope)
            
        # Adaptive LayerNorm cuối cùng dựa trên t_emb
        x = self.final_norm(x, t_emb)
        
        # Dự đoán v_t (Trường vận tốc)
        v_t = self.final_proj(x) # [B, T, 100]
        
        # Tẩy sạch vùng padding ở đầu ra cuối cùng
        if mask is not None:
            v_t = v_t.masked_fill(~mask.unsqueeze(-1), 0.0)
            
        return v_t

Writing models.py


In [8]:
%%writefile trainer.py
import os
import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
from torch import amp
from copy import deepcopy
import matplotlib.pyplot as plt
import librosa.display
import numpy as np
from tqdm import tqdm
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import csv
from dataset import SpeechProcessor


def load_viflow_weights(model, optimizer, path, device):
    if not os.path.exists(path):
        return 0, 0
    
    checkpoint = torch.load(path, map_location=device)
    state_dict = checkpoint['model_state_dict']
    model.load_state_dict(state_dict)
    
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        
    return checkpoint['epoch'], checkpoint['step'], checkpoint.get('ema_state_dict')

class EMA:
    def __init__(self, model_instance, beta=0.9999):
        self.model = deepcopy(model_instance)
        self.beta = beta
        self.model.eval()
        for param in self.model.parameters():
            param.requires_grad = False

    @torch.no_grad()
    def update(self, model):
        # Hỗ trợ cả model đơn và model bọc trong DDP
        model_params = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        ema_params = self.model.state_dict()
        
        for name, param in ema_params.items():
            if name in model_params:
                param.copy_(param * self.beta + model_params[name] * (1.0 - self.beta))
    
class ViFlowTrainer:
    def __init__(
        self, 
        model, 
        optimizer, 
        processor,
        engine,
        train_loader, 
        val_loader, 
        config, 
        rank=0, 
        start_step=0, 
        start_epoch=0, 
        ema_state=None,
        verbose=True
    ):
        
        self.rank = rank
        self.device = torch.device(f'cuda:{rank}')
        self.is_ddp = torch.distributed.is_initialized()
        self.is_main_process = (rank == 0)
        self.verbose = verbose and self.is_main_process
        
        self.processor = processor
        self.engine = engine
        
        self.model = model.to(self.device)
        self.raw_model = model.module if self.is_ddp else model
        
        self.step = start_step
        self.start_epoch = start_epoch

        if self.verbose:
            print(f"Khởi tạo chương trình huấn luyện: DDP {self.is_ddp}, Start step: {self.step}, Start epoch: {self.start_epoch}")
        
        self.grad_accum_steps = config.get("grad_accum_steps", 1)
        
        self.log_dir = config.get("log_dir", "logs")
        self.train_log_path = os.path.join(self.log_dir, "train_log.csv")
        self.val_log_path = os.path.join(self.log_dir, "val_log.csv")
        
        if self.is_main_process:
            self._init_logs()
            if self. verbose:
                print(f"Đã tạo thư mục chứa logs cho training và validation!")
            
        self.ema = EMA(self.raw_model, beta=config.get("ema_beta", 0.9999))
        if ema_state is not None:
            self.ema.model.load_state_dict(ema_state)
        if self.verbose:
            print("Đã khởi tạo mô hình EMA!")

        self.optimizer = optimizer
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.config = config
        
        self.use_amp = config["train"]["use_amp"]
        self.scaler = amp.GradScaler('cuda', enabled=self.use_amp)

    def _init_logs(self):
        """Khởi tạo thư mục và file CSV với tiêu đề cột"""
        if not os.path.exists(self.log_dir):
            os.makedirs(self.log_dir)
            
        # File log cho training (ghi theo step)
        if not os.path.exists(self.train_log_path):
            with open(self.train_log_path, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['epoch', 'step', 'loss'])
                
        # File log cho validation (ghi theo epoch/mốc validate)
        if not os.path.exists(self.val_log_path):
            with open(self.val_log_path, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(['epoch', 'step', 'val_loss', 'ema_val_loss'])

    def _write_csv(self, path, row):
        """Hàm phụ trợ ghi một dòng vào CSV"""
        if self.is_main_process:
            with open(path, 'a', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(row)

    def train_step(self, batch, ode=False, ode_samples=1):
        self.model.train()
        
        processed = self.processor.process_batch(batch)
        text_ids = processed["text_ids"].to(self.device)
        mels = processed["target_mels"].to(self.device)
        mel_mask = processed["mel_mask"].to(self.device)
        mel_lens = processed["target_mel_lens"].to(self.device)

        # Tao ma trận nhiễu, điều kiện, vector vận tốc và thời gian t
        xt, cond, ut, t, target_mask = self.engine.get_train_batch(mels, mel_lens, mel_mask)
        
        if self.step % self.grad_accum_steps == 0:
            self.optimizer.zero_grad()

        with amp.autocast("cuda", enabled=self.use_amp):
            vt = self.model(x=xt, cond=cond, text_ids=text_ids, t=t, mask=target_mask)
            loss = self.engine.compute_loss(vt, ut, target_mask) / self.grad_accum_steps

        # Backward
        if self.use_amp:
            # Luồng FP16 (amp)
            self.scaler.scale(loss).backward()
            
            if (self.step + 1) % self.grad_accum_steps == 0:
                self.scaler.unscale_(self.optimizer)
                clip_grad_norm_(self.model.parameters(), self.config['train']["grad_clip"])
                self.scaler.step(self.optimizer)
                self.scaler.update()
                if self.ema: self.ema.update(self.model)
        else:
            # Luồng FP32
            loss.backward()
            if (self.step + 1) % self.grad_accum_steps == 0:
                clip_grad_norm_(self.model.parameters(), self.config['train']["grad_clip"])
                self.optimizer.step()
                if self.ema: self.ema.update(self.model)

        self.step += 1
        
        if ode and self.is_main_process:
            self.sample_and_visualize(processed, self.step, ode_samples)
            
        return self._reduce_loss(loss * self.grad_accum_steps)

    @torch.no_grad()
    def validate(self):
        """
        Đánh giá mô hình trên tập validation.
        So sánh Loss giữa Online Model và EMA Model.
        """
        self.model.eval()
        self.ema.model.eval() # Model EMA luôn ở chế độ eval
        
        local_val_loss = 0.0
        local_ema_loss = 0.0
        num_batches = len(self.val_loader)
        
        print(f"🔍 Đang chạy Validation trên {len(self.val_loader)} batches...")

        for batch in self.val_loader:
            # 1. Tiền xử lý dữ liệu
            processed = self.processor.process_batch(batch)
            text_ids = processed["text_ids"].to(self.device)
            mels = processed["target_mels"].to(self.device)
            mel_lens = processed["target_mel_lens"].to(self.device)
            mel_mask = processed["mel_mask"].to(self.device)

            # 2. Lấy dữ liệu Flow Matching từ Engine
            xt, cond, ut, t, target_mask = self.engine.get_train_batch(mels, mel_lens, mel_mask)

            # 3. Tính Loss cho Online Model
            with torch.amp.autocast('cuda', enabled=self.use_amp):
                # Forward Online
                vt_online = self.model(x=xt, cond=cond, text_ids=text_ids, t=t, mask=~mel_mask)
                loss_online = self.engine.compute_loss(vt_online, ut, target_mask)
                local_val_loss += loss_online.item()

                # Forward EMA
                vt_ema = self.ema.model(x=xt, cond=cond, text_ids=text_ids, t=t, mask=~mel_mask)
                loss_ema = self.engine.compute_loss(vt_ema, ut, target_mask)
                local_ema_loss += loss_ema.item()

        if self.is_ddp and dist.is_initialized():
            metrics = torch.tensor([local_val_loss, local_ema_loss, float(num_batches)], device=self.device)
            dist.all_reduce(metrics, op=dist.ReduceOp.SUM)
            val_loss = metrics[0] / metrics[2]
            ema_val_loss = metrics[1] / metrics[2]
        else:
            val_loss = local_val_loss / num_batches
            ema_val_loss = local_ema_loss / num_batches

        print(f"Kết quả Validation:")
        print(f"   - Online Loss: {val_loss:.6f}")
        print(f"   - EMA Loss:    {ema_val_loss:.6f}")
        
        self.model.train()
        return {
            "val_loss": val_loss,
            "ema_val_loss": ema_val_loss
        }

    def fit(self, solve_ode=True, visualize_per_steps=200, validate_per_steps=200, write_loss_per_steps=200):
        total_epochs = self.config.get("num_epochs", 100)
        if self.verbose:
            print(f"Bắt đầu huấn luyện ViFlow từ Epoch {self.start_epoch} đến {total_epochs}. Step {self.step}")
            
        for epoch in range(self.start_epoch, total_epochs):
            # Xáo trộn dữ liệu theo epochs
            if self.is_ddp:
                self.train_loader.sampler.set_epoch(epoch)
            batches_to_skip = self.step % len(self.train_loader) if epoch == self.start_epoch else 0
            pbar = tqdm(
                self.train_loader, 
                desc=f"Epoch {epoch}", 
                disable=not self.is_main_process,
                dynamic_ncols=True
            )
            for i, batch in enumerate(pbar):
                if batches_to_skip > 0:
                    batches_to_skip -= 1
                    continue
                ode = self.is_main_process and (solve_ode) and (self.step % visualize_per_steps) == 0
                validate = self.step % validate_per_steps == 0 if validate_per_steps is not None else False
                
                loss = self.train_step(batch, ode=ode)
                pbar.set_postfix({
                    "loss": f"{loss:.4f}"
                })
                
                if self.step % write_loss_per_steps ==  0 and self.is_main_process:
                    self._write_csv(self.train_log_path, [epoch, self.step, f"{loss:.6f}"])

                if self.step % self.config['train']['save_interval'] == 0:
                    self.save_checkpoint(epoch, self.step, folder=self.config['train']['checkpoint_dir'])
                    
                if validate:
                    val_results = self.validate()
                    pbar.set_postfix({
                        "val_loss": f"{val_results['val_loss']:.4f}",
                        "ema_loss": f"{val_results['ema_val_loss']:.4f}"
                    })
                    self._write_csv(self.val_log_path, [
                        epoch, 
                        self.step, 
                        f"{val_results['val_loss']:.6f}", 
                        f"{val_results['ema_val_loss']:.6f}"
                    ])

            if self.is_main_process:
                if self.verbose:
                    print(f"\n--- Kết thúc Epoch {epoch} ---")
                val_results = self.validate()
                if self.verbose:
                    print(f"Validation epoch {epoch}, step {self.step}:\n\t - Val loss: {val_results['val_loss']}\n\t - EMA Loss: {val_results['ema_val_loss']}")
                
                # Lưu checkpoint riêng cho mỗi epoch để làm mốc (Backup) 
                self.save_checkpoint(epoch + 1, self.step, folder=self.config['train']['checkpoint_dir'])

        if self.verbose:
            print("Huấn luyện hoàn tất.")

    def save_checkpoint(self, epoch, step, folder="checkpoints"):
        """Lưu trữ cả model 'chiến đấu' và model 'hậu phương' EMA"""
        if not os.path.exists(folder):
            os.makedirs(folder)
            
        checkpoint = {
            'epoch': epoch,
            'step': step,
            'model_state_dict': self.model.module.state_dict() if hasattr(self.model, 'module') else self.model.state_dict(),
            'ema_state_dict': self.ema.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'config': self.config
        }
        path = os.path.join(folder, f"viflow_step_{step}.pt")
        torch.save(checkpoint, path)
        if self.verbose:
            print(f" Đã lưu checkpoint tại: {path}")

    def _reduce_loss(self, loss):
        """
        Gom Loss từ tất cả các GPU tham gia huấn luyện và tính trung bình cộng.
        Điều này giúp con số hiển thị trên thanh tiến trình phản ánh đúng 
        trạng thái của toàn bộ hệ thống.
        """
        if not self.is_ddp:
            return loss.item()

        dist_loss = loss.clone().detach()
        torch.distributed.all_reduce(dist_loss, op=torch.distributed.ReduceOp.SUM)
        avg_loss = dist_loss.item() / torch.distributed.get_world_size()
        return avg_loss

    @torch.no_grad()
    def sample_and_visualize(self, processed_batch, step, num_samples=2, folder="val_outputs"):
        self.ema.model.eval()
        if not os.path.exists(folder):
            os.makedirs(folder)

        text_ids = processed_batch["text_ids"][:num_samples]
        mels_gt = processed_batch["target_mels"][:num_samples]
        mel_lens = processed_batch["target_mel_lens"][:num_samples]
        mel_mask = processed_batch["mel_mask"][:num_samples]

        # 2. Infilling Scenario (30% Prompt)
        p_ratio = 0.3
        p_lens = (p_ratio * mel_lens).long()
        indices = torch.arange(mels_gt.size(1), device=self.device).unsqueeze(0)
        target_mask = (~mel_mask) & (indices >= p_lens.unsqueeze(1))
        m_exp = target_mask.unsqueeze(-1).float()

        cond = (1.0 - m_exp) * mels_gt * (~mel_mask).unsqueeze(-1).float()
        x0 = torch.randn_like(mels_gt)
        x0 = (1.0 - m_exp) * mels_gt + m_exp * x0

        if self.verbose:
            print(f"[Step {step}] Generating samples...")
        x_gen = self.engine.solve_ode(
            model=self.ema.model,
            x0=x0,
            steps=self.config['inference']['ode_steps'], 
            cond=cond,
            text_ids=text_ids,
            mask=target_mask,
            cfg_scale=1.5
        )
        self._plot_comparison(mels_gt, x_gen, mel_lens, p_lens, step, folder)

    def _plot_comparison(self, real_mels, fake_mels, lens, p_lens, step, folder):

        for i in range(real_mels.size(0)):
            actual_len = lens[i].item()
            prompt_end = p_lens[i].item()
            
            # Chuyển về numpy và cắt padding
            real = real_mels[i, :actual_len].cpu().numpy().T
            fake = fake_mels[i, :actual_len].cpu().numpy().T

            fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
            
            # Dùng chung vmax, vmin để màu sắc đồng nhất dễ so sánh
            vmin = min(real.min(), fake.min())
            vmax = max(real.max(), fake.max())
            # vmin, vmax = real.min(), real.max()

            # Vẽ Mel thật
            librosa.display.specshow(real, ax=ax1, y_axis='mel', x_axis='time', sr=24000, hop_length=256, vmax=vmax, vmin=vmin)
            ax1.axvline(x=prompt_end * 256 / 24000, color='red', linestyle='--', label='Prompt End')
            ax1.set_title(f"Real Mel (Step {step})")

            # Vẽ Mel giả
            librosa.display.specshow(fake, ax=ax2, y_axis='mel', x_axis='time', sr=24000, hop_length=256, vmax=vmax, vmin=vmin)
            ax2.axvline(x=prompt_end * 256 / 24000, color='red', linestyle='--', label='Prompt End')
            ax2.set_title(f"Generated Mel (EMA Model)")

            plt.tight_layout()
            plt.savefig(os.path.join(folder, f"viflow_val_{step}_sample_{i}.png"), dpi=150)
            plt.close()
            
            # diff = real - fake
            
            # # Lưu ma trận kết quả ra file .npy
            # diff_filename = os.path.join(folder, f"diff_prompt_step_{step}_sample_{i}.npy")
            # np.save(diff_filename, diff)

Writing trainer.py


In [9]:
%%writefile train_viflow.py
import os
import yaml
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader

# Import các thành phần nội bộ của sếp
from engine import ViFlowEngine
from dataset import VietnameseTokenizer, SpeechProcessor, ViFlowDataset, identity_collate
from models import ViFlowOTCFM
from trainer import ViFlowTrainer, load_viflow_weights

def load_config(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)

def main_worker(rank, world_size, config):
    """Tiến trình con chạy trên từng GPU"""
    try:
        is_ddp = world_size > 1
        device = torch.device(f'cuda:{rank}')
        torch.cuda.set_device(device)

        if is_ddp:
            os.environ['MASTER_ADDR'] = 'localhost'
            os.environ['MASTER_PORT'] = config['train']['port']
            dist.init_process_group("nccl", rank=rank, world_size=world_size)
        
        device = torch.device(f'cuda:{rank}')
        torch.cuda.set_device(device)
    
        # 2. Chuẩn bị "Vũ khí" (Model, Processor, Engine)
        tokenizer = VietnameseTokenizer()
        processor = SpeechProcessor(
            tokenizer=tokenizer,
            mel_cfg=config['mel'],
            device=device
        )
        engine = ViFlowEngine(sigma_min=config['train'].get('sigma_min', 1e-5))
    
        model = ViFlowOTCFM(
            dim=config['model']['hidden_dim'],
            depth=config['model']['num_dit_blocks'],
            head_dim=config['model']['head_dim'],
            heads=config['model']['num_heads'],
            text_dim=config['model']['text_dim'],
            text_mask_padding=config['model']['text_mask_padding'],
            mel_dim=config['model']['mel_dim'],
            vocab_size=tokenizer.vocab_size,
            text_layers=config['model']['text_layers'],
            pe_attn_head=config['model']['pe_attn_head'],
            dropout=config['model']['dropout']
        ).to(device)
    
        optimizer = torch.optim.AdamW(
            model.parameters(), 
            lr=float(config['train']['learning_rate']),
            betas=(0.9, 0.95),
            weight_decay=float(config['train']['weight_decay'])
        )
    
        start_epoch = 0
        start_step = 0
        ema_state = None
        
        # Đường dẫn checkpoint
        ckpt_path = config['train'].get('resume_path', '')
        if ckpt_path and os.path.exists(ckpt_path):
            if rank == 0:
                print(f"📂 Đang nạp checkpoint cho tất cả GPU từ: {ckpt_path}")
            
            start_epoch, start_step, ema_state = load_viflow_weights(
                model, optimizer, ckpt_path, device
            )
        else:
            if rank == 0:
                print("🚀 Không tìm thấy checkpoint. Bắt đầu huấn luyện mới.")

        if is_ddp:
            model = DDP(model, device_ids=[rank], output_device=rank)
    
        # Thiết lập Dữ liệu (DDP Sampler)
        train_dataset = ViFlowDataset(
            dataset_dirs=config['data']['dataset_dirs'], 
            sample_rate=config['mel']['sample_rate'],
            exclude_csv_path=config['data']['val_sample_path']
        )
        
        if is_ddp:
            train_sampler = torch.utils.data.distributed.DistributedSampler(
                train_dataset, num_replicas=world_size, rank=rank, shuffle=True
            )
        else: train_sampler = None
    
        train_loader = DataLoader(
            train_dataset, 
            batch_size=config['train']['batch_size'], 
            sampler=train_sampler,
            shuffle=False, 
            num_workers=config['train']['num_workers'], 
            pin_memory=True,
            collate_fn=identity_collate
        )
    
        val_dataset = ViFlowDataset(
            dataset_dirs=config['data']['dataset_dirs'], 
            sample_rate=config['mel']['sample_rate'],
            include_csv_path=config['data']['val_sample_path']
        )
        
        if is_ddp:
            val_sampler = torch.utils.data.distributed.DistributedSampler(
                val_dataset, num_replicas=world_size, rank=rank, shuffle=False # Val không cần xáo trộn
            )
        else: val_sampler = None
        
        val_loader = DataLoader(
            val_dataset, 
            batch_size=config['train']['batch_size'], 
            sampler=val_sampler, # Chia việc cho các GPU
            shuffle=False, 
            num_workers=config['train']['num_workers'],
            collate_fn=identity_collate
        )
    
        # 6. Kích hoạt Trainer
        trainer = ViFlowTrainer(
            model=model,
            optimizer=optimizer,
            processor=processor,
            engine=engine,
            train_loader=train_loader,
            val_loader=val_loader,
            config=config,
            rank=rank,
            start_step=start_step,
            start_epoch=start_epoch,
            ema_state=ema_state
        )
    
        # Bắt đầu cuộc hành trình
        trainer.fit()
    except Exception as e:
        print(f"❌ GPU {rank} sập tiệm vì lỗi: {e}")
        raise e
    finally:
        if torch.distributed.is_initialized():
            dist.destroy_process_group()

def load_config(config_path):
    """Đọc file cấu hình và trả về dictionary"""
    if not os.path.exists(config_path):
        raise FileNotFoundError(f"❌ Không tìm thấy file cấu hình tại: {config_path}")
        
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    return config

if __name__ == "__main__":
    config = load_config('configs.yaml')
    world_size = torch.cuda.device_count()

    if world_size > 1:
        print(f"🔥 Chế độ Đa GPU: Khởi chạy DDP với {world_size} GPU.")
        mp.spawn(main_worker, args=(world_size, config), nprocs=world_size, join=True)
    else:
        print("🚀 Chế độ Đơn GPU: Chạy trực tiếp trên cuda:0.")
        # Nếu chỉ có 1 GPU, gọi thẳng hàm không cần spawn để dễ debug
        main_worker(0, 1, config)

Writing train_viflow.py


In [ ]:
!python train_viflow.py

🚀 Chế độ Đơn GPU: Chạy trực tiếp trên cuda:0.
🚀 Không tìm thấy checkpoint. Bắt đầu huấn luyện mới.
✅ Dataset ready: 29930 samples (Filters applied via CSV)
✅ Dataset ready: 167 samples (Filters applied via CSV)
Khởi tạo chương trình huấn luyện: DDP False, Start step: 0, Start epoch: 0
Đã tạo thư mục chứa logs cho training và validation!
Đã khởi tạo mô hình EMA!
Bắt đầu huấn luyện ViFlow từ Epoch 0 đến 100. Step 0
Epoch 0:   0%|                            | 0/936 [00:40<?, ?it/s, loss=46.1768]🔍 Đang chạy Validation trên 6 batches...


In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import os
# import glob

# # 1. Tìm file .npy mới nhất trong thư mục đầu ra
# folder = "val_outputs"
# npy_files = glob.glob(os.path.join(folder, "*.npy"))

# if not npy_files:
#     print("❌ Không tìm thấy file .npy nào trong thư mục!")
# else:
#     # Lấy file mới nhất để kiểm tra
#     latest_file = max(npy_files, key=os.path.getctime)
#     print(f"📂 Đang kiểm tra file: {latest_file}")

#     # 2. Load ma trận sai lệch (R - F)
#     diff_matrix = np.load(latest_file)
    
#     # Giả sử ma trận có dạng [Mel_bins, Time_frames]
#     mel_bins, total_frames = diff_matrix.shape
    
#     # 3. Lấy 30% đầu tiên (Vùng Prompt)
#     # Lưu ý: Nếu sếp đã lưu file này từ vùng prompt rồi thì diff_matrix chính là 30% đó.
#     # Nếu sếp lưu toàn bộ ma trận, hãy dùng dòng dưới đây:
#     prompt_limit = int(0.3 * total_frames)
#     prompt_diff = diff_matrix[:, :prompt_limit]

#     # 4. Tính toán thống kê định lượng
#     mae = np.mean(np.abs(prompt_diff))
#     max_diff = np.max(np.abs(prompt_diff))
#     mse = np.mean(prompt_diff**2)

#     print(f"\n📊 Thống kê sai lệch vùng Prompt:")
#     print(f"   - Sai số tuyệt đối trung bình (MAE): {mae:.8f}")
#     print(f"   - Sai số bình phương trung bình (MSE): {mse:.8f}")
#     print(f"   - Sai lệch lớn nhất (Max Diff): {max_diff:.8f}")

#     # 5. Trực quan hóa bằng Heatmap
#     plt.figure(figsize=(12, 6))
#     # Dùng abs để thấy rõ cường độ sai lệch dù âm hay dương
#     plt.imshow(np.abs(prompt_diff), aspect='auto', origin='lower', cmap='magma')
#     plt.colorbar(label='Absolute Difference Intensity')
#     plt.title(f"Bản đồ sai lệch vùng Prompt (30% đầu)\nFile: {os.path.basename(latest_file)}")
#     plt.xlabel("Time Frames")
#     plt.ylabel("Mel Filterbank Bins")
#     plt.show()

#     # 6. Kiểm tra xem có bằng 0 tuyệt đối không
#     if mae < 1e-7:
#         print("✅ Vùng Prompt khớp hoàn toàn (Sai số xấp xỉ 0).")
#     else:
#         print("⚠️ Cảnh báo: Vùng Prompt đã bị thay đổi giá trị trong quá trình Inference.")